# MedFusion-RAFT — Quick Start Notebook

This notebook demonstrates the MedFusion-RAFT pipeline using **mock providers** (no GPU required).

> **⚠️ Research Prototype** — This system is NOT a clinical tool. Generated answers must NOT be used for diagnosis or treatment.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.models.schemas import (
    MedicalQuestion, RetrievedPassage, GeneratedAnswer,
    AnswerabilityLabel, VerificationResult
)
from src.models.base import ModelConfig
from src.models.mock_providers import MockGeneratorProvider, MockVerifierProvider
from src.verification.claim_extractor import ClaimExtractor
from src.verification.citation_validator import CitationValidator
from src.verification.entailment import MockEntailmentChecker
from src.verification.pipeline import VerificationPipeline, VerificationPipelineConfig
from src.verification.decision import DecisionEngine, DecisionPolicyConfig

print('All imports successful!')

## 1. Initialize Components

In [ ]:
# Generator (mock MedGemma)
generator = MockGeneratorProvider(ModelConfig(model_id='mock-medgemma'))
generator.load()

# Verifier (mock Qwen)
verifier = MockVerifierProvider(ModelConfig(model_id='mock-qwen'))
verifier.load()

# Verification pipeline
ver_pipeline = VerificationPipeline(
    config=VerificationPipelineConfig(use_llm_verification=False),
    verifier_provider=verifier,
    entailment_checker=MockEntailmentChecker(),
)

# Decision engine
decision_engine = DecisionEngine(DecisionPolicyConfig())

print('Components initialized.')

## 2. Simulate Retrieved Evidence

In [ ]:
passages = [
    RetrievedPassage(
        passage_id='passage_001', doc_id='doc_001',
        text='Metformin is a biguanide used as first-line treatment for type 2 diabetes. '
             'It decreases hepatic glucose production and increases insulin sensitivity.',
        section='Pharmacology', score=0.92, retrieval_method='hybrid_rrf', rank=1,
    ),
    RetrievedPassage(
        passage_id='passage_002', doc_id='doc_001',
        text='Contraindications include severe renal impairment (eGFR < 30), '
             'metabolic acidosis, and hypersensitivity.',
        section='Contraindications', score=0.85, retrieval_method='hybrid_rrf', rank=2,
    ),
]

for p in passages:
    print(f'[{p.passage_id}] score={p.score:.2f}: {p.text[:60]}...')

## 3. Generate Answer

In [ ]:
question = 'What is the mechanism of action of metformin?'
context = '\n\n'.join(f'[P{i+1}] {p.text}' for i, p in enumerate(passages))

gen_output = generator.generate_answer(question, context)

answer = GeneratedAnswer(
    answer_text=gen_output.text.split('ANSWER:')[-1].split('CITATIONS:')[0].strip()
        if 'ANSWER:' in gen_output.text else gen_output.text,
    answerability=AnswerabilityLabel.ANSWERABLE,
    confidence=0.8,
    model_id='mock-medgemma',
    generation_time_ms=gen_output.generation_time_ms,
)

print(f'Answer ({gen_output.output_tokens} tokens, {gen_output.generation_time_ms:.0f}ms):')
print(answer.answer_text)

## 4. Verify Answer

In [ ]:
ver_result = ver_pipeline.verify(question, answer, passages)

print(f'Verified: {ver_result.verified}')
print(f'Supported claims: {ver_result.supported_claims}')
print(f'Unsupported claims: {ver_result.unsupported_claims}')
print(f'Contradicted claims: {ver_result.contradicted_claims}')
print(f'Overall score: {ver_result.overall_score:.3f}')

## 5. Decision (Accept / Revise / Abstain)

In [ ]:
decision = decision_engine.decide(
    answer=answer,
    verification=ver_result,
    num_evidence_passages=len(passages),
)

print(f'Decision: {decision.decision.value.upper()}')
print(f'Composite score: {decision.score:.3f}')
print(f'Reasoning: {decision.reasoning}')

## 6. Evaluation Metrics

In [ ]:
from src.evaluation.answer_metrics import compute_f1, compute_rouge_l

reference = 'Metformin decreases hepatic glucose production and increases insulin sensitivity.'
predicted = answer.answer_text

f1 = compute_f1(predicted, reference)
rouge = compute_rouge_l(predicted, reference)

print(f'Token F1: {f1:.3f}')
print(f'ROUGE-L: {rouge:.3f}')

---

*MedFusion-RAFT — Research prototype. Not for clinical use.*